# Types of Machine Learning: interactive demos

Run each cell with **Shift + Enter**. Every chart is interactive: drag to rotate, scroll to zoom, hover for details.

## 1. Dimensionality reduction: digits in 3D

Each handwritten digit is an 8 x 8 image, so it is stored as **64 numbers** (64 columns). We cannot plot 64 dimensions.
PCA squeezes those 64 columns into 3 while keeping as much of the structure as possible.

**Try:** rotate the cloud until the digits separate. Which digits overlap the most?

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
import plotly.express as px

digits = load_digits()                 # 1,797 images, each as 64 numbers
print(digits.data.shape)               # (rows, columns)

xyz = PCA(n_components=3).fit_transform(digits.data)   # 64 columns -> 3 columns
print(xyz.shape)

fig = px.scatter_3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
                    color=digits.target.astype(str),
                    category_orders={"color": [str(d) for d in range(10)]},
                    labels={"color": "Digit"}, height=650)
fig.update_traces(marker_size=3)
fig.show()

## 2. Clustering: how many groups?

The students below have only two columns, IQ and CGPA, and **no labels**. K-Means splits them into *k* groups.
We have to choose *k* ourselves.

**Try:** move the slider. Which value of *k* matches the groups you can see by eye?

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import KMeans

rng = np.random.default_rng(11)
centres = [(120, 8.8), (118, 6.2), (85, 8.5)]                 # example data: 3 hidden groups
X = np.vstack([np.column_stack([rng.normal(iq, 5, 30), rng.normal(cg, 0.35, 30)]) for iq, cg in centres])
scaled = (X - X.mean(0)) / X.std(0)                            # put IQ and CGPA on the same scale

fig = go.Figure()
for k in range(2, 7):                                          # one coloured scatter per k
    groups = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(scaled)
    fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers", visible=(k == 3),
                             marker=dict(size=10, color=groups, colorscale="Turbo"), name=f"k = {k}"))

steps = [dict(method="update", label=str(k), args=[{"visible": [j == i for j in range(5)]}])
         for i, k in enumerate(range(2, 7))]
fig.update_layout(sliders=[dict(active=1, steps=steps, currentvalue=dict(prefix="Number of groups k = "))],
                  xaxis_title="IQ", yaxis_title="CGPA", height=550, template="simple_white")
fig.show()